In [7]:
from pydantic import BaseModel, Field, GetCoreSchemaHandler
from pydantic_core import CoreSchema, core_schema
from typing import Any

class OverZero(BaseModel):
    value: int = Field(gt=0, description="必须大于0的整数")

    @classmethod
    def __get_pydantic_core_schema__(
        cls, source_type: Any, handler: GetCoreSchemaHandler
    ) -> CoreSchema:
        # 获取 OverZero 模型自身的校验模式
        inner_schema = handler(cls)
        # 定义转换函数：int -> OverZero 实例
        def ensure_over_zero(v: Any) -> Any:
            if isinstance(v, int):
                return cls(value=v)
            return v  # 假设已经是 OverZero 实例

        # 在校验之前插入转换逻辑
        return core_schema.no_info_before_validator_function(
            ensure_over_zero,
            inner_schema,
        )

# ---------- 使用示例 ----------
from pydantic import validate_call

@validate_call
def func(a: OverZero):
    print("✅ passed:", a.value)

func(5)               # ✅ passed: 5
func(OverZero(value=10))  # ✅ passed: 10

try:
    func(0)
except Exception as e:
    print("❌", e)     # 验证错误: value must be greater than 0

✅ passed: 5
✅ passed: 10
❌ 1 validation error for func
0.value
  Input should be greater than 0 [type=greater_than, input_value=0, input_type=int]
    For further information visit https://errors.pydantic.dev/2.12/v/greater_than
